## Handling some issues with the dataset and reducing features

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

df=pd.read_csv("../data/processed/initial/second_innings_merged.csv")
df.head()

,Unnamed: 0,match_id,innings,team_batting,team_bowling,over_number,ball_number,total_runs,is_wicket,is_wide_ball,is_no_ball,current_score,wickets_lost,season_id,match_date,city,venue,toss_winner,team1,team2,toss_decision,match_winner,result,first_innings_runs,target_runs,runs_left,legal_ball,total_legal_balls,overs_completed,CRR,balls_left,overs_left,RRR
0,0,335982,2,1,6,0,0,1,False,False,False,1,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,222,True,1,0.166667,6.0,119,19.833333,11.193277
1,1,335982,2,1,6,0,1,1,False,True,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,False,1,0.166667,12.0,119,19.833333,11.142857
2,2,335982,2,1,6,0,2,0,False,False,False,2,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,221,True,2,0.333333,6.0,118,19.666667,11.237288
3,3,335982,2,1,6,0,3,1,False,False,False,3,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,220,True,3,0.500000,6.0,117,19.500000,11.282051
4,4,335982,2,1,6,0,4,1,False,False,False,4,0,2008,2008-04-18,Bangalore,M Chinnaswamy Stadium,1,1,6,field,6,win,222,223,219,True,4,0.666667,6.0,116,19.333333,11.327586


Sorting the rows and checking consistency

In [2]:
df = df.sort_values(['match_id', 'Unnamed: 0']).reset_index(drop=True)

# The engineered columns should be internally consistent
print((df.runs_left == df.target_runs - df.current_score).all())         
print((df.balls_left == 120 - df.total_legal_balls).all())               
print(df.groupby('match_id').current_score.apply(lambda s: s.is_monotonic_increasing).all())

True
True
True


Normalization of venue names(Claude generated)

In [3]:
# Kaggle's match data spells the same ground in several ways, because the names differ across seasons and sources. 
# A model sees each spelling as a separate venue, which splits one stadium's history into thin slices. 
# Dubai and Sharjah have no city in the source, so it is filled from the venue.

VENUE_MAP = {
    'Arun Jaitley Stadium': 'Arun Jaitley Stadium, Delhi',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium, Delhi',
    'Brabourne Stadium': 'Brabourne Stadium, Mumbai',
    'Dr DY Patil Sports Academy': 'Dr DY Patil Sports Academy, Mumbai',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam',
    'Eden Gardens': 'Eden Gardens, Kolkata',
    'Himachal Pradesh Cricket Association Stadium': 'Himachal Pradesh Cricket Association Stadium, Dharamsala',
    'M Chinnaswamy Stadium': 'M Chinnaswamy Stadium, Bengaluru',
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium, Bengaluru',
    'MA Chidambaram Stadium': 'MA Chidambaram Stadium, Chepauk, Chennai',
    'MA Chidambaram Stadium, Chepauk': 'MA Chidambaram Stadium, Chepauk, Chennai',
    'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur': 'Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh',
    'Maharashtra Cricket Association Stadium': 'Maharashtra Cricket Association Stadium, Pune',
    'Subrata Roy Sahara Stadium': 'Maharashtra Cricket Association Stadium, Pune',
    'Punjab Cricket Association IS Bindra Stadium': 'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh',
    'Punjab Cricket Association Stadium, Mohali': 'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh',
    'Rajiv Gandhi International Stadium': 'Rajiv Gandhi International Stadium, Uppal, Hyderabad',
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi International Stadium, Uppal, Hyderabad',
    'Sardar Patel Stadium, Motera': 'Narendra Modi Stadium, Ahmedabad',
    'Sawai Mansingh Stadium': 'Sawai Mansingh Stadium, Jaipur',
    'Wankhede Stadium': 'Wankhede Stadium, Mumbai',
    'Sheikh Zayed Stadium': 'Zayed Cricket Stadium, Abu Dhabi',
}
df['venue'] = df['venue'].replace(VENUE_MAP)
df['city'] = df['city'].fillna(df['venue'].map({
    'Dubai International Cricket Stadium': 'Dubai',
    'Sharjah Cricket Stadium': 'Sharjah'}))

print(df.venue.nunique())
df.city.isna().sum()

36


np.int64(0)

Also, we need to drop the ties or no result since they are not helpful for the model prediction

In [4]:
last = df.groupby('match_id').tail(1)    # the final row of each match

bad_ids = last[
    (last.result != 'win') |                                          # tie or no result
    ((last.team_batting == last.match_winner) & (last.runs_left > 0)) # chasing team "won" but never reached the target
].match_id

df = df[~df.match_id.isin(bad_ids)]

Exclude the defunct franchises (Rising Pune Supergiant, Gujarat Lions, Kochi Tuskers Kerala, Pune Warriors). 
Deccan Chargers is kept as part of Sunrisers Hyderabad (id 2), since they represent the same region

In [5]:
# A match has a batting side and a bowling side, so both columns are checked.
# Otherwise a CSK vs Pune Warriors game would leave the defunct team in team_bowling.
# The ids are kept for now, because the target and the tie filter compare team_batting with match_winner (both ids)

DEFUNCT_IDS = {4, 5, 1414, 1419}

defunct = df.team_batting.isin(DEFUNCT_IDS) | df.team_bowling.isin(DEFUNCT_IDS)
df = df[~defunct]

print(df.match_id.nunique(), len(df))

1063 123213


Now comes the edge cases like infinite values in RRR/CRR, 

In [6]:
df = df[df['legal_ball']]    # one row per real delivery

g = df.groupby('match_id')
df['wickets_left']    = 10 - df['wickets_lost']
df['runs_last_30']    = df['current_score'] - g['current_score'].shift(30).fillna(0)
df['wickets_last_30'] = df['wickets_lost']  - g['wickets_lost'].shift(30).fillna(0)

df = df[(df.runs_left > 0) & (df.balls_left > 0) & (df.wickets_lost < 10)]

Finally build the target variable to predict what happened in the match

In [7]:
df['won'] = (df['team_batting'] == df['match_winner']).astype(int)

Map the team ids to team names (done after the target is built, because `won` compares the ids)

In [8]:
TEAM_ID_TO_NAME = {
    1: "Royal Challengers Bengaluru",
    2: "Sunrisers Hyderabad",
    3: "Mumbai Indians",
    6: "Kolkata Knight Riders",
    129: "Chennai Super Kings",
    134: "Rajasthan Royals",
    252: "Delhi Capitals",
    494: "Punjab Kings",
    614: "Lucknow Super Giants",
    615: "Gujarat Titans",
}

df['team_batting'] = df['team_batting'].map(TEAM_ID_TO_NAME)
df['team_bowling'] = df['team_bowling'].map(TEAM_ID_TO_NAME)

print(df[['team_batting', 'team_bowling']].isna().sum())   # should be 0
print(df.won.mean())                                        # should be around 0.51 - 0.53

team_batting    0
team_bowling    0
dtype: int64
0.5155752904949047


Drop the irrelevant columns

In [9]:
df = df.drop(columns=[
    'Unnamed: 0', 'innings',                                      # index, constant
    'match_winner', 'result',                                     # label source: leakage
    'team1', 'team2', 'toss_winner', 'toss_decision',             # redundant / weak
    'total_legal_balls', 'overs_completed', 'overs_left',         # copies of balls_left
    'over_number', 'ball_number',                                 # same information again
    'total_runs', 'is_wicket', 'is_wide_ball', 'is_no_ball', 'legal_ball',  # single-ball events
    'first_innings_runs',                                         # target_runs = this + 1
    'wickets_lost',                                               # wickets_left carries it
])

df.head()

,match_id,team_batting,team_bowling,current_score,season_id,match_date,city,venue,target_runs,runs_left,CRR,balls_left,RRR,wickets_left,runs_last_30,wickets_last_30,won
0,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,1,2008,2008-04-18,Bangalore,"M Chinnaswamy Stadium, Bengaluru",223,222,6.0,119,11.193277,10,1.0,0.0,0
2,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,2,2008,2008-04-18,Bangalore,"M Chinnaswamy Stadium, Bengaluru",223,221,6.0,118,11.237288,10,2.0,0.0,0
3,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,3,2008,2008-04-18,Bangalore,"M Chinnaswamy Stadium, Bengaluru",223,220,6.0,117,11.282051,10,3.0,0.0,0
4,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,4,2008,2008-04-18,Bangalore,"M Chinnaswamy Stadium, Bengaluru",223,219,6.0,116,11.327586,10,4.0,0.0,0
5,335982,Royal Challengers Bengaluru,Kolkata Knight Riders,4,2008,2008-04-18,Bangalore,"M Chinnaswamy Stadium, Bengaluru",223,219,4.8,115,11.426087,10,4.0,0.0,0


Saving the final dataset before ML

In [ ]:
df.to_csv("../data/processed/reduced/Final_data.csv", index=False)